In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Dense)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [5]:
# Sample data
data = [
    ('i am learning', 'je suis en train d apprendre'),
    ('he is running', 'il court'),
    ('she is happy', 'elle est heureuse'),
    ('i am happy', 'je suis heureuse')
]

input_texts = [x[0] for x in data]
target_texts = ['<start>'+x[1]+'<end>' for x in data]

In [6]:
# Tokenization

# encoder tokenization
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

# decoder tokenization
target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)

In [7]:
# Padding
max_input_length = max([len(x) for x in input_sequences])
max_target_length = max([len(x) for x in target_sequences])

input_sequences = pad_sequences(input_sequences, maxlen=max_input_length, padding='post')
target_sequences = pad_sequences(target_sequences, maxlen=max_target_length, padding='post')

In [8]:
decoder_target_data = np.zeros_like(target_sequences)
for i in range(target_sequences.shape[0]):
    decoder_target_data[i, :-1] = target_sequences[i, 1:]

In [10]:
# Build model
vocab_size_input = len(input_tokenizer.word_index)
vocab_size_target = len(target_tokenizer.word_index)
embedding_dim = 50
latent_dim = 100

# Encoder
encoder_inputs = Input(shape=(max_input_length,))
encoder_embedding = tf.keras.layers.Embedding(vocab_size_input+1, embedding_dim, mask_zero=True)(encoder_inputs)

encoder_lstm = tf.keras.layers.LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]
print(encoder_states)

# Decoder
decoder_inputs = Input(shape=(max_target_length,))
decoder_embedding = tf.keras.layers.Embedding(vocab_size_target+1, embedding_dim, mask_zero=True)(decoder_inputs)

decoder_lstm = tf.keras.layers.LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

decoder_dense = tf.keras.layers.Dense(vocab_size_target+1, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)
print(decoder_outputs)

[<KerasTensor shape=(None, 100), dtype=float32, sparse=False, ragged=False, name=keras_tensor_10>, <KerasTensor shape=(None, 100), dtype=float32, sparse=False, ragged=False, name=keras_tensor_11>]
<KerasTensor shape=(None, 8, 14), dtype=float32, sparse=False, ragged=False, name=keras_tensor_18>


In [11]:
# Compile model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 3, 50)     │        450 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 3)         │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 8, 50)     │        700 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 100),     │     60,400 │ embedding_1[0][0… │
│                     │ (None, 100),      │            │ not_equal_1[0][0] │
│                     │ (None, 100)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 8, 100),  │     60,400 │ embedding_2[0][0… │
│                     │ (None, 100),      │            │ lstm_1[0][1],     │
│                     │ (None, 100)]      │            │ lstm_1[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 8, 14)     │      1,414 │ lstm_2[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 123,364 (481.89 KB)

 Trainable params: 123,364 (481.89 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# Train model
model.fit(
    [input_sequences, target_sequences],
    decoder_target_data,
    batch_size=2,
    epochs=50,
    validation_split=0.2
)

Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 1s/step - accuracy: 0.1176 - loss: 2.6365 - val_accuracy: 0.2000 - val_loss: 2.6255
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.2353 - loss: 2.6259 - val_accuracy: 0.2000 - val_loss: 2.6177
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 0.1765 - loss: 2.6165 - val_accuracy: 0.2000 - val_loss: 2.6101
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.2353 - loss: 2.6064 - val_accuracy: 0.2000 - val_loss: 2.6014
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - accuracy: 0.2353 - loss: 2.5937 - val_accuracy: 0.2000 - val_loss: 2.5914
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.2353 - loss: 2.5801 - val_accuracy: 0.2000 - val_loss: 2.5799
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.2353 - loss: 2.5638 - val_accuracy: 0.2000 - val_loss: 2.5661
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.2353 - loss: 2.5439 - val_accuracy: 0.2000 - val_loss: 2.

In [14]:
# Encoder interface
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder interface
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs, state_h, state_c = decoder_lstm(decoder_embedding, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states)


In [17]:
# Prediction function
reverse_target_index = {i: word for word, i in target_tokenizer.word_index.items()}

def decode_sequence(input_seq):

    states_value = encoder_model.predict(input_seq)

    start_token = target_tokenizer.word_index.get('<start>')

    if start_token is None:
        start_token = target_tokenizer.word_index.get('start')

    end_token = target_tokenizer.word_index.get('<end>')

    if end_token is None:
        end_token = target_tokenizer.word_index.get('end')

    target_seq = np.array([[start_token]])

    stop_condition = False
    decoded_sentence = ""

    while not stop_condition:

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value
        )

        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        sampled_word = reverse_target_index.get(
            sampled_token_index, ''
        )

        if (
            sampled_token_index == end_token or
            len(decoded_sentence.split()) > max_target_length
        ):
            stop_condition = True
        else:
            decoded_sentence += " " + sampled_word

        target_seq = np.array([[sampled_token_index]])

        states_value = [h, c]

    return decoded_sentence

In [24]:
# test
input_sentence = "i am you"
input_sequence = input_tokenizer.texts_to_sequences([input_sentence])
input_sequence = pad_sequences(input_sequence, maxlen=max_input_length, padding='post')
decode_sequence(input_sequence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step


' suis en train d apprendre'

In [23]:
type(input_sequence)

numpy.ndarray